In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, display
from src.to_html import cell, heading, image, paragraph, table

In [2]:
train_type_name_mapping = {
    "AKN": "AKN Eisenbahn",
    "ALX": "alex",
    "ASTB": "Autoschleuse Tauernbahn",
    "BE": "Bentheimer Eisenbahn",
    "BEX": "Bördeexpress",
    "BOB": "Bodensee-Oberschwaben-Bahn",
    "BRB": "Bayerische Regiobahn",
    "Bus": "Bus",
    "CAT": "City Airport Train",
    "CB": "CB",
    "CBG": "CBG",
    "CJX": "Cityjet xpress",
    "D": "Schnellzug",
    "DBK": "DBK",
    "DRV": "Schnellzug für Reiseveranstalter",
    "EC": "EuroCity",
    "ECE": "ECE",
    "EN": "EuroNight",
    "ENO": "ENNO",
    "ERB": "Eurobahn",
    "ES": "ES",
    "EST": "Eurostar",
    "EVB": "EVB",
    "FEX": "Flughafen-Express",
    "FLX": "Flixtrain",
    "GABW": "Arverio Baden-Württemberg",
    "GABY": "Arverio Bayern",
    "HBX": "Harz-Berlin-Express",
    "HLB": "Hessische Landesbahn",
    "HzL": "Hohenzollerische Landesbahn AG",
    "IC": "Intercity",
    "ICE": "Intercity-Express",
    "IRE": "Interregio-Express",
    "ME": "Metronom",
    "MEX": "Metropolexpress",
    "MRB": "Mitteldeutsche Regiobahn",
    "N": "N",
    "NBE": "Nordbahn Eisenbahngesellschaft",
    "NJ": "Nightjet",
    "NOB": "Nord-Ostsee-Bahn",
    "NWB": "NordWestBahn",
    "OE": "Ostdeutsche Eisenbahn",
    "OPB": "OPB",
    "R": "Regionalzug",
    "RB": "Regionalbahn",
    "RDC": "RDC",
    "RE": "Regional-Express",
    "REX": "Regional-Express",
    "RGJ": "Regiojet",
    "RJ": "Railjet",
    "RJX": "Railjet xpress",
    "RRB": "RRB",
    "RS": "Regio-S-Bahn Bremen/Niedersachsen",
    "RT": "RegioTram",
    "RTB": "Rurtalbahn GmbH",
    "S": "S-Bahn",
    "SAB": "Schwäbische Alb-Bahn",
    "SB": "Städtebahn Sachsen",
    "SBB": "Schweizerische Bundesbahnen",
    "SC": "SuperCity",
    "SE": "Städtebahn Sachsen",
    "STB": "STB",
    "STN": "STN",
    "SVG": "SVG",
    "SWE": "Südwestdeutsche Landesverkehrs-Gesellschaft",
    "TEL": "TEL",
    "TGV": "TGV",
    "TL": "TL",
    "TLX": "TLX",
    "TRI": "TRI",
    "UEX": "UEX",
    "VIA": "VIA",
    "WB": "WESTbahn",
    "WEST": "WESTbahn",
    "WFB": "Westfalenbahn",
    "ag": "ag",
    "erx": "erixX - Der Heidesprinter",
}

In [3]:
save_dir = Path("../stats/data")
data_dir = Path("../monthly_processed_data")
files = sorted(data_dir.glob("data-*.parquet"))[-2:]

df_list = []
for file in files:
    df_list.append(pd.read_parquet(file)[["station_name", "delay_in_min", "is_canceled", "train_type"]])
df = pd.concat(df_list, ignore_index=True)

df["train_type_name"] = df["train_type"].map(train_type_name_mapping)

print(f"Loaded {len(df):,} records from {len(files)} files")
print(f"Files: {[f.name for f in files]}")

Loaded 29,407,028 records from 2 files
Files: ['data-2025-11.parquet', 'data-2025-12.parquet']


In [4]:
content = cell(
    heading("Wie verteilen sich Verspätungen auf verschiedene Zuggattungen?", level=1)
    + paragraph("Hier kann man die verschiedenen Zuggattungen über alle Bahnhöfe hinweg miteinander vergleichen.")
    + paragraph(
        'In den Daten der Deutschen Bahn hat jeder Zughalt das Attribut "train_type", welches sich mit Zuggattung übersetzen lässt. '
        "Dies gibt an um welche Art von Zug es sich handelt (ICE, Re oder S-Bahn), aber sie beschreibt auch manchmal welches Bahnunternehmen den Zug betreibt (SWB, NWB usw.)."
    )
    + paragraph(
        'Soweit ich das verstehe, ist der Betreiber aber nicht immer in der Zuggattung, da z.B. der Betreiber "RRX" als Zuggattungen nie auftaucht. '
        'Wenn jemand weiß, was es mit "train_type" genau auf sich hat und wie man es gut interpretieren kann, könnt ihr mir gerne eine Email schreiben oder ein issue bei GitHub erstellen.'
    )
)
display(HTML(content))

In [5]:
# Calculate statistics per train type
stats = (
    df[~df["is_canceled"]]
    .groupby("train_type_name")
    .agg(
        {
            "delay_in_min": "mean",
            "train_type": lambda x: ", ".join(sorted(set(x))),
        }
    )
    .reset_index()
)

cancellation_sample_size_df = (
    df.groupby("train_type_name")
    .agg({"is_canceled": "mean", "station_name": "size"})
    .rename(
        columns={
            "is_canceled": "cancellation_rate",
            "station_name": "sample_size",
        }
    )
)
stats = stats.merge(cancellation_sample_size_df, on="train_type_name")
stats_sorted = stats.sort_values("sample_size", ascending=False)

In [6]:
# Create bar chart for top 15 train types
top_15 = stats_sorted.head(15)
labels = [f"{name} ({train_type})" for name, train_type in zip(top_15["train_type_name"], top_15["train_type"])]

fig, ax1 = plt.subplots(figsize=(16, 10))

width = 0.35
x = range(len(labels))

bars1 = ax1.bar(
    [i - width / 2 for i in x],
    top_15["delay_in_min"],
    width,
    label="Durchschnittliche Verspätung",
    color="b",
    alpha=0.7,
)

ax2 = ax1.twinx()

bars2 = ax2.bar(
    [i + width / 2 for i in x], top_15["cancellation_rate"], width, label="Ausfallquote", color="r", alpha=0.7
)

ax1.set_xlabel("Name der Zuggattung")
ax1.set_ylabel("Durchschnittliche Verspätung (Minuten)")
ax2.set_ylabel("Ausfallquote")
plt.title("Durchschnittliche Verspätung und Ausfallquote der 15 größten Zuggattungen")

ax1.set_xticks(x)
ax1.set_xticklabels(labels, rotation=45, ha="right")


def add_value_labels(bars, ax, x_offset=0):
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2.0 + x_offset,
            height,
            f"{height:.2f}",
            ha="center",
            va="bottom",
        )


add_value_labels(bars1, ax1)
add_value_labels(bars2, ax2, x_offset=0.07)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.tight_layout()
plt.savefig(save_dir / "top_15_verspaetung_und_ausfallquote.png", dpi=150, bbox_inches="tight")
plt.close()

content = cell(
    heading("Durchschnittliche Verspätung und Ausfallquote der 15 häufigsten Zuggattungen", level=2)
    + image("data/top_15_verspaetung_und_ausfallquote.png", "Verspätung und Ausfallquote der 15 größten Zuggattungen")
)
display(HTML(content))

In [7]:
# Create table with all train types
table_df = stats_sorted.copy()
table_df["delay_in_min"] = table_df["delay_in_min"].round(2)
table_df["cancellation_rate"] = table_df["cancellation_rate"].round(2)
table_df = table_df.rename(
    columns={
        "train_type_name": "Zuggattung Name",
        "train_type": "Zuggattung Abkürzung",
        "delay_in_min": "Durchschn. Verspätung [min]",
        "cancellation_rate": "Ausfallquote",
        "sample_size": "Anzahl Zughalte",
    }
)

content = cell(heading("Alle Zuggattungen", level=2) + table(table_df))
display(HTML(content))

Zuggattung Name ⇕,Durchschn. Verspätung [min] ⇕,Zuggattung Abkürzung ⇕,Ausfallquote ⇕,Anzahl Zughalte ⇕
S-Bahn,2.63,S,0.05,13472172
Regionalbahn,3.1,RB,0.03,5058872
Regional-Express,5.19,"RE, REX",0.03,3321203
Bus,0.83,Bus,0.01,831833
Hessische Landesbahn,4.16,HLB,0.04,709761
Bayerische Regiobahn,2.75,BRB,0.02,437311
NordWestBahn,4.78,NWB,0.03,405849
VIA,5.06,VIA,0.04,381208
Intercity-Express,10.95,ICE,0.04,364200
ag,3.33,ag,0.02,361447


In [8]:
start_date = files[0].stem.replace("data-", "")
end_date = files[-1].stem.replace("data-", "")
content = cell(
    f'<p style="font-size: 0.8em; text-align: center;">Quelle: <a href="https://github.com/piebro/deutsche-bahn-data/blob/main/notebooks/zuggattung.ipynb">Berechnet</a> auf Basis von <a href="https://github.com/piebro/deutsche-bahn-data">gesammelten Daten</a> von der Deutschen Bahn vom {start_date} bis {end_date}.</p>'
)
display(HTML(content))